# Pipeline de Regresión Ridge

Este cuaderno implementa un pipeline de preprocesamiento y modelado utilizando una regresión lineal regularizada (**Ridge Regression**).

## 1. Importación de Librerías

Cargamos las librerías necesarias de análisis de datos y modelización de scikit-learn.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

## 2. Cargar Datos

Cargamos el dataset `vehicles.csv`. Si no existe localmente, se intentará descargar desde Kaggle.

In [ ]:
if os.path.exists('vehicles.csv'):
    print("Cargando datos desde 'vehicles.csv'...")
    df = pd.read_csv('vehicles.csv', encoding='latin1', on_bad_lines='skip', low_memory=False)
else:
    print("El archivo 'vehicles.csv' no existe localmente. Intentando descargar de Kaggle...")
    try:
        os.environ['KAGGLE_API_TOKEN'] = "KGAT_a0fcdeed3125aab3d652c9bc808ee163"
        import kaggle
        kaggle.api.dataset_download_files('austinreese/craigslist-carstrucks-data', path='.', unzip=True)
        print("¡Descarga completada con éxito!")
        df = pd.read_csv('vehicles.csv', encoding='latin1', on_bad_lines='skip', low_memory=False)
    except Exception as e:
        print(f"Error al descargar de Kaggle: {e}")
        print("Por favor, coloca 'vehicles.csv' en este directorio y vuelve a intentarlo.")
        raise FileNotFoundError("vehicles.csv no encontrado.")

## 3. Limpieza y Filtrado de Datos

Eliminamos filas duplicadas y columnas innecesarias de baja relevancia. Filtramos precios atípicos, odómetros fuera de rangos razonables y años extremos.

In [ ]:
print("Preprocesando y limpiando el dataset...")
df_clean = df.drop_duplicates()
cols_irrelevantes = ['id', 'url', 'region_url', 'image_url', 'description',
                     'VIN', 'county', 'posting_date', 'lat', 'long']
df_clean = df_clean.drop(columns=[c for c in cols_irrelevantes if c in df_clean.columns])

# Filtrado comercial
df_clean = df_clean[(df_clean['price'] > 500) & (df_clean['price'] <= 150000)]
df_clean = df_clean[df_clean['odometer'].isna() | ((df_clean['odometer'] >= 1) & (df_clean['odometer'] <= 500000))]
df_clean = df_clean[df_clean['year'].isna() | ((df_clean['year'] >= 1980) & (df_clean['year'] <= 2027))]

## 4. División de Conjuntos (Entrenamiento y Prueba)

Separamos el set en características `X` y variable a predecir `y` (`price`). Dividimos con un 20% para test y 80% para entrenamiento.

In [ ]:
X = df_clean.drop('price', axis=1)
y = df_clean['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identificación automática de tipos de columnas
columnas_numericas = X_train.select_dtypes(include=['int64', 'float64']).columns
columnas_categoricas = X_train.select_dtypes(include=['object', 'category']).columns

## 5. Sub-Pipelines de Preprocesamiento

- **Numéricas**: Imputamos nulos con la mediana de los valores, luego estandarizamos.
- **Categóricas**: Imputamos nulos con el valor 'unknown' y aplicamos codificación One-Hot.

In [ ]:
# Sub-pipeline numérico
pipeline_numerico = Pipeline(steps=[
    ('imputador', SimpleImputer(strategy='median')),
    ('escalador', StandardScaler())
])

# Sub-pipeline categórico
pipeline_categorico = Pipeline(steps=[
    ('imputador', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('codificador', OneHotEncoder(handle_unknown='ignore'))
])

# Ensamblado final
preprocesador = ColumnTransformer(
    transformers=[
        ('num', pipeline_numerico, columnas_numericas),
        ('cat', pipeline_categorico, columnas_categoricas)
    ])

## 6. Pipeline Completo y Entrenamiento

Unimos el preprocesador con el regresor `Ridge` y entrenamos.

In [ ]:
pipeline_ridge = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', Ridge(alpha=1.0))
])

print("Limpiando nulos, estandarizando, codificando y entrenando Ridge...")
pipeline_ridge.fit(X_train, y_train)
print("¡Entrenamiento completado sin errores!")